<a href="https://colab.research.google.com/github/jimhopgtu/google-ai-agents-daily-content-tool/blob/main/Daily_Relevant_Content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools.google_search_tool import GoogleSearchTool
from google.adk.tools import FunctionTool
from google.genai import types
import json
from pydantic import BaseModel, Field
from typing import List
import os
from google.colab import userdata, drive
from datetime import datetime
import pytz
eastern_tz = pytz.timezone('America/New_York')

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [2]:

# This pulls the secret you just created and sets it as an environment variable
# Most Google SDKs (including ADK) look for "GOOGLE_API_KEY" automatically.
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

print("✅ API Key successfully loaded into environment!")

# from IPython.core.display import display, HTML
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)


# Ensure Drive is mounted correctly
drive.mount('/content/drive', force_remount=True)

print("Executed at:", datetime.now(eastern_tz))

✅ API Key successfully loaded into environment!
Mounted at /content/drive
Executed at: 2025-12-19 15:30:48.881648-05:00


In [3]:
search_instruction = """
You are a Research Scout for an Analytics Leader.
Your goal is to provide a balanced mix of content . For every run, you MUST use the search tool to find:

1. THE LATEST (24h): Top 3 industry-shifting news (e.g., Anthropic, OpenAI, Google).
2. THE ARCHITECTURE: Top 1 technical blog posts from engineering-heavy companies
   (e.g., MongoDB, Pinecone, Meta Engineering) that discuss 'why' or 'how'—not just 'what'.
3. THE STACK: Top 2 recent updates from the modern BI stack (e.g., dbt, Snowflake, Databricks, BigQuery, Looker, Atlan, PowerBI, Tableau)
4. THE LOCAL: 1 AI event in the NYC area or a major remote global summit.
5. Top 1 Obsidian plug in or use case that are new and could be useful.

## CRITICAL CONSTRAINTS
<Constraints>
- ANTI-FLUFF: If results are 'marketing fluff', refine keywords to include 'technical deep dive'.
- OUTPUT FORMAT: Return ONLY the URL and a 1-2 sentence summary. No full articles.
- MULTIMEDIA: YouTube videos are acceptable.
</Constraints>

## OUTPUT FORMAT (MANDATORY)
For every single item you find, you MUST follow this exact format:
- **Title**: [Name of the article/video]
- **Source URL**: [Insert the full direct link here]
- **Summary**: [1-2 sentences of why this matters for a data leader]

## CRITICAL RULES
- NEVER provide a news item without a corresponding URL.
- If you find a great story but the URL is missing from your tool output, do not include the story.
"""

# Define the data structure as a Python list/dictionary
LEADER_CONTEXT_DATA = {
    "ANALYTICS_LEADER_CONTEXT": [
        {
            "Category": "Modeling",
            "Shift": "Causal Inference (MMM/MTA), Advanced Models",
            "Action": "Prioritize Experimentation and Causal Strategy (A/B testing, incrementality)"
        },
        {
            "Category": "Architecture",
            "Shift": "Semantic Layer is SOT, often led by **Knowledge Engineer**",
            "Action": "Architect **AI Trust** and robust **Data Governance**"
        },
        {
            "Category": "Role & Skills",
            "Shift": "Analyst as Prompt Engineer/Consultant",
            "Action": "Coach for **Business Acumen** (the 'Why') and focus on recommendations"
        },
        {
            "Category": "Specialization",
            "Shift": "Deep expertise in one major stack (e.g., GCP, Azure)",
            "Action": "Standardize and Optimize the chosen stack to maximize value"
        },
        {
            "Category": "Analyst Profile",
            "Shift": "**Hybrid Role** (Business SME + Data Engineering + **Knowledge Engineering**)",
            "Action": "Redefine career path, mandate **DE fundamentals** and context structuring"
        },
        {
            "Category": "Governance",
            "Shift": "Mandatory focus on **Data Governance** and **AI TRiSM**",
            "Action": "Audit AI outputs, enforce lineage, and ensure ethical compliance"
        },
        {
            "Category": "Speed",
            "Shift": "Shift to **Real-Time** and **Edge Analytics**",
            "Action": "Invest in Modern Architecture (e.g., Data Mesh) for streaming data processing"
        },
        {
            "Category": "Leadership",
            "Shift": "Highest value in Human-Centric/Soft Skills and organizational influence",
            "Action": "Cultivate critical thinking, emotional intelligence, and **cross-department bridge-building** to remove data silos"
        }
    ]
}

# Convert it to a pretty-printed string ONLY when you need to feed it to the Agent
context_string = json.dumps(LEADER_CONTEXT_DATA, indent=2)



relevance_instruction = f"""
## ROLE
You are a Quality Controller for an Analytics Leader.

## DATA SOURCES
1. **NEW ARTICLES**: {{raw_news_data}}
2. **CURRENT VAULT**: {{approved_stories?}}

## STRATEGIC CONTEXT
{context_string}

## YOUR TASK
For each article in raw_news_data:
1. Assign a score (1-5).
2. Categorize it (e.g., "AI Tools", "Data Engineering", "Market News").
3. Suggest a brief 'action' for the leader (e.g., "Monitor for implementation", "Share with team").
4. Return the data using these EXACT keys: title, url, summary, score, category, action.

## OUTPUT RULES (STRICT COMPLIANCE REQUIRED)
- Return ONLY the raw JSON object.
- DO NOT use markdown code blocks (no triple backticks).
- DO NOT include "```json" or "```".
- Start your response with {{ and end it with }}.  <-- Double braces fix the f-string error
- If you include any text other than raw JSON, the system will crash.
"""


from pydantic import BaseModel, Field, AliasChoices
from typing import List, Optional

# This replaces 'ScoredArticle' to match the system expectations
class EvaluatedArticle(BaseModel):
    title: str
    # 'validation_alias' allows the AI to say 'url' OR 'source_url' without crashing
    url: str = Field(validation_alias=AliasChoices('url', 'source_url'))
    summary: str
    score: int
    # Making these 'Optional' with default values prevents "Missing Field" crashes
    category: Optional[str] = "General"
    action: Optional[str] = "Review for strategy"

class RelevanceResponse(BaseModel):
    evaluated_articles: List[EvaluatedArticle]


print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-19 15:30:51.613279-05:00


In [4]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the story is finished and no more changes are needed."""
    return {"status": "approved", "message": "Story approved. Exiting refinement loop."}


print("✅ exit_loop function created.")
print("Executed at:", datetime.now(eastern_tz))

✅ exit_loop function created.
Executed at: 2025-12-19 15:30:55.431987-05:00


In [5]:

# 1. Search Agent
search_agent = Agent(
    name="news_finder",
    model="gemini-2.0-flash", # Use 2.0 for speed/tool use gemini-2.0-flash gemini-1.5-flash or gemini-1.5-flash-8b
    tools=[GoogleSearchTool()],
    instruction=search_instruction,
    output_key="raw_news_data"
)

# 2. Relevance Agent
relevance_agent = Agent(
    name="relevance_evaluator",                  # 1. Added required 'name'
    model=Gemini(model_id="gemini-2.0-flash-exp"),
    instruction=relevance_instruction            # 2. Changed 'instructions' to 'instruction'
)

# 3. Loop Agent
story_refinement_loop = LoopAgent(
    name="StoryRefinementLoop",
    sub_agents=[search_agent, relevance_agent],
    # The SDK usually looks for a single condition string referencing the state
    max_iterations=3
)

# 4. Reporting Agent
reporting_agent = Agent(
    name="reporting_agent",
    model="gemini-2.0-flash",
    instruction="""
    ## TASK
    1. Read the articles stored in {{approved_stories?}}.
    2. If the vault is empty, simply state "No highly relevant news found for today."
    3. If articles exist, format them into a professional Markdown report.

    ## FORMAT REQUIREMENTS
    - Use ## Headers for different categories.
    - Use **Bolding** for key takeaways.
    - Ensure every Title is a clickable [Markdown Link](URL).
    - Provide a "Why this matters" section for each article.
    """
)

print("Executed at:", datetime.now(eastern_tz))

Executed at: 2025-12-19 15:30:57.043493-05:00


In [16]:
# Replace the entire cell that starts with "# 1. Start with the Search Agent"
# This version uses external variable storage instead of runner.state

# ====================
# EXTERNAL STORAGE
# ====================
# This dictionary persists across agent runs in the same notebook session
APPROVED_STORIES_STORAGE = {}

# 1. Start with the Search Agent
runner = InMemoryRunner(agent=search_agent)

# 2. RUN SEARCH
print("🔍 Searching...")
search_events = await runner.run_debug(
    "Find news for the analytics leader.",
    session_id="daily_report_session"
)

# 3. SWAP THE AGENT MANUALLY
runner.agent = relevance_agent

# 4. RUN RELEVANCE
print("⚖️ Evaluating...")
import re
import json

relevance_events = await runner.run_debug(
    "Evaluate the data in raw_news_data.",
    session_id="daily_report_session"
)

# Get the raw text from the last event
raw_ai_text = relevance_events[-1].content.parts[0].text

print("\n--- DEBUG: RAW AI RESPONSE START ---")
print(raw_ai_text[:1000])
if len(raw_ai_text) > 1000:
    print("... [TRUNCATED for brevity] ...")
print("--- DEBUG: RAW AI RESPONSE END ---\n")

# --- MANUAL JSON CLEANING & VALIDATION ---
clean_json = re.sub(r'^```json\s*|```$', '', raw_ai_text, flags=re.MULTILINE | re.DOTALL).strip()

try:
    # Parse the JSON
    parsed_json = json.loads(clean_json)

    # Handle if AI returned array directly vs wrapped in object
    if isinstance(parsed_json, list):
        print("💡 AI sent a list; wrapping it in 'evaluated_articles' key...")
        data_to_validate = {"evaluated_articles": parsed_json}
    elif "articles" in parsed_json:
        print("💡 AI used 'articles' key; renaming to 'evaluated_articles'...")
        data_to_validate = {"evaluated_articles": parsed_json["articles"]}
    elif "evaluated_articles" in parsed_json:
        data_to_validate = parsed_json
    else:
        raise ValueError("Could not find articles in AI response")

    # Validate with Pydantic
    validated_data = RelevanceResponse.model_validate(data_to_validate)

    # 🎯 STORE IN EXTERNAL VARIABLE (not runner.state)
    APPROVED_STORIES_STORAGE["approved_stories"] = validated_data.model_dump()

    print(f"✅ Success! Validated {len(validated_data.evaluated_articles)} articles.")
    print(f"📊 Stored in APPROVED_STORIES_STORAGE")

except Exception as e:
    print(f"❌ Error parsing Relevance JSON: {e}")
    import traceback
    traceback.print_exc()

# 5. SWAP TO REPORTER
runner.agent = reporting_agent

# 6. RUN REPORT - Pass the data directly in the prompt
print("📊 Reporting...")

# Create a formatted string of the approved stories
approved_stories_text = json.dumps(
    APPROVED_STORIES_STORAGE.get("approved_stories", {}),
    indent=2
)

final_report = await runner.run_debug(
    f"""Generate the markdown report from this data:

{approved_stories_text}

Format it professionally with headers, links, and clear sections.""",
    session_id="daily_report_session"
)

# --- COST TRACKER ---
print("\n" + "="*30)
print("💰 COST & USAGE REPORT")
print("="*30)

total_input = 0
total_output = 0

# Collect usage from all event lists
all_events = search_events + relevance_events + final_report

for event in all_events:
    if hasattr(event, 'usage_metadata') and event.usage_metadata:
        total_input += event.usage_metadata.prompt_token_count
        total_output += event.usage_metadata.candidates_token_count

# Gemini 2.0 Flash pricing (adjust if using different models)
input_cost = (total_input / 1_000_000) * 0.10
output_cost = (total_output / 1_000_000) * 0.40

print(f"Input Tokens:  {total_input:,}")
print(f"Output Tokens: {total_output:,}")
print(f"Estimated Cost: ${input_cost + output_cost:.5f}")
print("="*30)

# --- SHOW SUMMARY ---
print("\n" + "="*30)
print("📋 STORAGE SUMMARY")
print("="*30)
if APPROVED_STORIES_STORAGE.get("approved_stories"):
    article_count = len(APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"])
    print(f"✅ Stored {article_count} evaluated articles")
    print("📁 Data available for markdown export")
else:
    print("⚠️ No stories in storage")
print("="*30)

🔍 Searching...

 ### Created new session: daily_report_session

User > Find news for the analytics leader.


news_finder > Okay, I will find the news for the analytics leader according to the defined guidelines. Here are the search queries I'll use:

Here's the news for the analytics leader:

**THE LATEST (24h)**

- **Title**: OpenAI and Anthropic to Launch AI Age Prediction Feature
- **Source URL**: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHF84mxp8MCeijeG04WOy6Ry_6yJNUZl0Fi26Mfce_9u1JbskrboGb8iGrQvMSUP9egGJFECMka7wU6MNSh0DcqIoEHoNB2_aWG9yyOuQeCGYBYr2FT9AnmTXht5A==
- **Summary**: OpenAI and Anthropic are launching AI-powered features to predict user age, aiming to enhance online safety for young users by tailoring AI interactions and automatically detecting/shutting down non-compliant accounts. These initiatives involve adapting ChatGPT guidelines for users under 18 and identifying subtle clues in conversations to detect minors.

- **Title**: Google Labs Unveils a Powerful AI Assistant CC: Automatically Sends Emails Every Morning to Manage Your Gmail, Calendar, and

relevance_evaluator > {"articles": [
  {
    "title": "OpenAI and Anthropic to Launch AI Age Prediction Feature",
    "url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHF84mxp8MCeijeG04WOy6Ry_6yJNUZl0Fi26Mfce_9u1JbskrboGb8iGrQvMSUP9egGJFECMka7wU6MNSh0DcqIoEHoNB2_aWG9yyOuQeCGYBYr2FT9AnmTXht5A==",
    "summary": "OpenAI and Anthropic are launching AI-powered features to predict user age, aiming to enhance online safety for young users by tailoring AI interactions and automatically detecting/shutting down non-compliant accounts. These initiatives involve adapting ChatGPT guidelines for users under 18 and identifying subtle clues in conversations to detect minors.",
    "score": 4,
    "category": "AI Governance & Safety",
    "action": "Monitor for industry best practices related to AI TRiSM and ethical AI development. Share with governance team."
  },
  {
    "title": "Google Labs Unveils a Powerful AI Assistant CC: Automatically Sends Emails Every Morning to M

In [17]:
# This cell exports the report to a markdown file in Google Drive

# 1. Setup the Path
today_date = datetime.now(eastern_tz).strftime("%Y-%m-%d_%H%M")
filename = f"{today_date}_Analytics_Report.md"
save_path = "/content/drive/MyDrive/AI/Obsidian Vault/daily news/"

# 2. Create folder if it's missing
if not os.path.exists(save_path):
    print(f"Creating missing directory: {save_path}")
    os.makedirs(save_path, exist_ok=True)

full_path = os.path.join(save_path, filename)

# 3. Extract and Clean the Report
try:
    # Look back for the actual text content
    final_report_text = ""
    for event in reversed(final_report):
        if event.content and event.content.parts:
            text_parts = [p.text for p in event.content.parts if p.text]
            if text_parts:
                final_report_text = "\n".join(text_parts)
                break

    if not final_report_text:
        raise ValueError("No report content found in final_report events")

    # Remove the ```markdown code block wrappers
    clean_report = re.sub(r'^```(?:markdown)?\n?|```$', '', final_report_text.strip(), flags=re.MULTILINE)

    # Add metadata header
    header = f"""---
title: Analytics Leader Daily Report
date: {datetime.now(eastern_tz).strftime("%Y-%m-%d")}
generated: {datetime.now(eastern_tz).strftime("%Y-%m-%d %H:%M %Z")}
---

"""

    final_content = header + clean_report

    # 4. Write the file and FORCE a sync
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(final_content)
        f.flush()  # Force write to the OS buffer
        os.fsync(f.fileno())  # Force write to the disk

    print(f"✅ Success! File physically written.")
    print(f"📂 Filename: {filename}")
    print(f"📍 Full Path: {full_path}")
    print(f"📏 File Size: {len(final_content)} characters")

    # Show first few lines as preview
    print("\n--- PREVIEW (first 500 chars) ---")
    print(final_content[:500])
    if len(final_content) > 500:
        print("...")
    print("--- END PREVIEW ---\n")

except Exception as e:
    print(f"❌ Error during saving: {e}")
    import traceback
    traceback.print_exc()

    # Fallback: Try to save raw data
    print("\n🔄 Attempting fallback save with raw storage data...")
    try:
        if APPROVED_STORIES_STORAGE.get("approved_stories"):
            fallback_content = "# Analytics Report (Fallback)\n\n"
            fallback_content += f"Generated: {datetime.now(eastern_tz).strftime('%Y-%m-%d %H:%M %Z')}\n\n"

            for article in APPROVED_STORIES_STORAGE["approved_stories"]["evaluated_articles"]:
                fallback_content += f"## [{article['title']}]({article['url']})\n\n"
                fallback_content += f"**Score:** {article['score']}/5 | **Category:** {article['category']}\n\n"
                fallback_content += f"{article['summary']}\n\n"
                fallback_content += f"**Action:** {article['action']}\n\n"
                fallback_content += "---\n\n"

            with open(full_path, "w", encoding="utf-8") as f:
                f.write(fallback_content)
                f.flush()
                os.fsync(f.fileno())

            print(f"✅ Fallback save successful!")
            print(f"📂 Filename: {filename}")
    except Exception as fallback_error:
        print(f"❌ Fallback also failed: {fallback_error}")

# 5. Final verification
print("\n🔍 File verification:")
!ls -lh "{full_path}"

✅ Success! File physically written.
📂 Filename: 2025-12-19_1620_Analytics_Report.md
📍 Full Path: /content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-19_1620_Analytics_Report.md
📏 File Size: 6915 characters

--- PREVIEW (first 500 chars) ---
---
title: Analytics Leader Daily Report
date: 2025-12-19
generated: 2025-12-19 16:20 EST
---

## Analytics Leader News Report

### AI Governance & Safety

-   **[OpenAI and Anthropic to Launch AI Age Prediction Feature](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHF84mxp8MCeijeG04WOy6Ry_6yJNUZl0Fi26Mfce_9u1JbskrboGb8iGrQvMSUP9egGJFECMka7wU6MNSh0DcqIoEHoNB2_aWG9yyOuQeCGYBYr2FT9AnmTXht5A==)**
    *   Summary: OpenAI and Anthropic are launching AI-powered features to predi
...
--- END PREVIEW ---


🔍 File verification:
-rw------- 1 root root 6.8K Dec 19 21:20 '/content/drive/MyDrive/AI/Obsidian Vault/daily news/2025-12-19_1620_Analytics_Report.md'


In [82]:
# !rm -rf /content/drive